In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here
import os
from torchvision.datasets import ImageFolder
import torchvision.transforms as transforms
from torch.utils.data import DataLoader


transform_train = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.Resize((32,32)),
    transforms.ToTensor()
])

transform_test = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.Resize((32,32)),
    transforms.ToTensor()
])

train_path = os.path.join(path, "PlantVillage/train")
test_path = os.path.join(path, "PlantVillage/test")

train_dataset = ImageFolder(train_path,transform_train )
test_dataset = ImageFolder(test_path,transform_test)

train_dataloader = DataLoader(train_dataset, 16, shuffle = True)
test_dataloader = DataLoader(test_dataset, 16, shuffle = False)



# Write your code here


In [ ]:
  import matplotlib.pyplot as plt
  import numpy as np

  # Define mean & std for denormalization (EfficientNet Preprocessing)


  # Display 5 images
  fig, axes = plt.subplots(1, 5, figsize=(15, 5))

  imgs_indices = [270,0,1223,1001,1600]

  for i in range(5):
      img, label = train_dataset[imgs_indices[i]]  # Load image & label

      # Convert tensor to numpy for visualization
      img_np = img.numpy().transpose(1, 2, 0)  # (C, H, W) → (H, W, C)

      # Denormalize the image
      img_np = np.clip(img_np, 0, 1)

      # Show image
      axes[i].imshow(img_np)
      axes[i].set_title(f'Class: {label}')
      axes[i].axis('off')

  plt.show()

In [ ]:
# Write your code here
import torch.nn as nn
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, 1, 1)
        self.conv2 = nn.Conv2d(16, 32, 3, 1,1)
        self.conv3 = nn.Conv2d(32,64, 3, 1, 1)
        self.conv4 = nn.Conv2d(64,128, 3, 1, 1)
        self.conv5= nn.Conv2d(128, 256, 3, 1, 1)
        self.pool = nn.MaxPool2d(2,2)
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(256, 100)
        self.fc2 = nn.Linear(100, 50)
        self.fc3 = nn.Linear(50, 3)


    def forward(self,x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)
        # x = nn.BatchNorm2d(x)

        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool(x)
        # x = nn.BatchNorm2d(x)

        x = self.conv3(x)
        x = self.relu(x)
        x = self.pool(x)
        # x = nn.BatchNorm2d(x)

        x = self.conv4(x)
        x = self.relu(x)
        x = self.pool(x)
        # x = nn.BatchNorm2d(x)

        x = self.conv5(x)
        x = self.relu(x)
        x = self.pool(x)
        # x = nn.BatchNorm2d(x)

        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.relu(x)

        x = self.fc2(x)
        x = self.relu(x)

        outputs = self.fc3(x)
        return outputs




In [ ]:
# Write your code here

# TO DO


def train_one_epoch(model, optimizer, criterion,train_loader, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total_samples = 0


    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss+= loss.item()

        predictions = outputs.argmax(dim=1)
        correct+= (predictions == labels).sum().item()
        total_samples += labels.shape[0]

    avg_loss = total_loss/len(train_loader)
    accuracy = correct/total_samples
    return avg_loss, accuracy

def validate_one_epoch(model, criterion, test_loader, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total_samples = 0
    with torch.no_grad():  # Disable gradient calculation

        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            predictions = outputs.argmax(dim=1)
            correct+= (predictions == labels).sum().item()
            total_samples += labels.shape[0]

            total_loss+= loss.item()

    avg_loss = total_loss/len(test_loader)
    accuracy = correct/total_samples

    return avg_loss, accuracy

In [ ]:
# Write your code here

import torch
import torch.optim as optim

from torchvision import models

num_classes = 3  # replace with number of classes in your dataset

# Load pretrained EfficientNet
  # or b1-b7 if you want bigger

# Replace the classifier (fc) to match your number of classes

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()
model = CNN()
model = model.to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)



In [ ]:
# TO DO

num_epochs = 5

print(device)
train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []
for epoch in range(num_epochs):
    avg_loss_train, train_accuracy = train_one_epoch(model, optimizer, criterion, train_dataloader, device)

    avg_loss_validation, val_accuracy = validate_one_epoch(model, criterion, test_dataloader, device)
    train_losses.append(avg_loss_train)
    train_accuracies.append(train_accuracy)
    val_losses.append(avg_loss_validation)
    val_accuracies.append(val_accuracy)

    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_loss_train:.4f}, Train accuracy: {train_accuracy:.4f} ,Val Loss: {avg_loss_validation:.4f}, Val Accuracy: {val_accuracy:.4f}')


In [ ]:
# TO DO

import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, num_epochs+1), val_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()


In [ ]:
# Write your code here
